## Retrieval-Augmented Generation (RAG) Document QA System

##This project demostrates a simple RAG pipeline that answers questions from a document using vector search and Large Language Model (LLM).
Technologies Used:
-Python
-FAISS (Vector Search)
-TF-IDF Embeddings
-Groq API
-Llama 3.3 LLM

In [1]:
##Upload the text document
with open("legal_sample_document.txt","r") as file:
    document = file.read()
print(document)

Employee termination requires a written notice of 30 days unless a serious breach occurs.

Confidentiality clauses require employees to protect company information during and after employment.

Payment terms specify that employees will receive salary on the last working day of each month.

Termination without notice may occur in cases of fraud or misconduct.


In [2]:
##Creating chunks
chunks = document.split("\n\n")

for i, chunk in enumerate(chunks):
    print(f"Chunk{i}:{chunk}")

Chunk0:Employee termination requires a written notice of 30 days unless a serious breach occurs.
Chunk1:Confidentiality clauses require employees to protect company information during and after employment.
Chunk2:Payment terms specify that employees will receive salary on the last working day of each month.
Chunk3:Termination without notice may occur in cases of fraud or misconduct.


In [3]:
##Create embeddings for chunks
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
vectorizer.fit_transform(chunks)

chunk_embeddings = vectorizer.fit_transform(chunks)

In [4]:
##Creating query
query = "What is the notice period for employee termination?"

##Converting query into embedding
query_embedding = vectorizer.transform([query])

In [5]:
##Calculate similarity
from sklearn.metrics.pairwise import cosine_similarity

similarity_scores = cosine_similarity(query_embedding,chunk_embeddings)
print(similarity_scores)

[[0.38167378 0.         0.14323496 0.22221478]]


In [6]:
best_chunk_index = similarity_scores.argmax()
print(chunks[best_chunk_index])

Employee termination requires a written notice of 30 days unless a serious breach occurs.


## FAISS Building

In [7]:
##Importing FAISS library
import faiss
import numpy as np

In [8]:
##Convert embeddings to Numpy array
##toarray()-converts sparse matrix to array
##float32-FAISS requires this format
embeddings_array = chunk_embeddings.toarray().astype("float32")
embeddings_array

array([[0.30641714, 0.        , 0.        , 0.30641714, 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.30641714,
        0.        , 0.        , 0.30641714, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.2415828 , 0.        , 0.30641714,
        0.19558209, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.        , 0.30641714, 0.        , 0.30641714,
        0.        , 0.2415828 , 0.        , 0.        , 0.        ,
        0.        , 0.30641714, 0.        , 0.        , 0.        ,
        0.30641714],
       [0.        , 0.29333723, 0.29333723, 0.        , 0.        ,
        0.29333723, 0.29333723, 0.29333723, 0.        , 0.        ,
        0.29333723, 0.        , 0.        , 0.23127043, 0.29333723,
        0.        , 0.        , 0.29333723, 0.        , 0.        ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.     

In [9]:
##Create FAISS index
##dimension-length of each embedding vector
##IndexFlatL2-similarity search index
dimension = embeddings_array.shape[1]
index = faiss.IndexFlatL2(dimension)

In [10]:
##Add vectors to FAISS
index.add(embeddings_array)

In [11]:
##Convert query to vector
query_vector = query_embedding.toarray().astype("float32")

In [12]:
##Search the vector database
k = 2

distances, indices = index.search(query_vector,k)
print(indices)

[[0 3]]


In [13]:
##Retrieve the actual text
for i in indices[0]:
    print(chunks[i])

Employee termination requires a written notice of 30 days unless a serious breach occurs.
Termination without notice may occur in cases of fraud or misconduct.


## GenAI Mini Project

In [14]:
##Retrive top chunks
k = 3 
distances, indices = index.search(query_vector,k)

##Combine retrieved chunks
retrieved_chunks = [chunks[i] for i in indices[0]]

context = " ".join(retrieved_chunks)
print("Retrieved Context:\n")
print(context)

Retrieved Context:

Employee termination requires a written notice of 30 days unless a serious breach occurs. Termination without notice may occur in cases of fraud or misconduct. Payment terms specify that employees will receive salary on the last working day of each month.


In [15]:
##Simulate an Answer Generator
def simple_answer(query, context):
    print("\nQuestion:", query)
    print("\nAnswer based on retrieved context:")
     #extract most relevant sentence
sentences = context.split(".")
print(sentences[0])

Employee termination requires a written notice of 30 days unless a serious breach occurs


In [16]:
import sys
print(sys.version)

3.13.9 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 19:09:58) [MSC v.1929 64 bit (AMD64)]


## AI Document Question Answering System using RAG

In [17]:
!pip install groq

In [18]:
from groq import Groq

#Replace with your Groq API key
client = 
Groq(api_key = "YOUR_API_KEY_HERE")

In [29]:
##Generate answer with LLM
def rag_answer(query, context):

    prompt = f"""
You are an AI assistant.

Answer the question using the provided context.

Context:
{context}

Question:
{query}

Answer clearly.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    return response.choices[0].message.content

In [30]:
answer = rag_answer(query,context)
print(answer)

The notice period for employee termination is 30 days, unless a serious breach occurs, such as fraud or misconduct, in which case termination may occur without notice.
